# Experiment 0: decide how long adaptation should run

Eight reference pilots follow one predefined full-update recipe per base/task through 20,000 successful updates. Their milestones are 0, 250, 1,000, 2,500, 5,000, 10,000 and 20,000.

The provisional main horizon is 5,000 updates. Inspect whether later behavior changes materially, together with cost and coverage, before freezing the main configs. A 5,000-update point from a 20,000-update schedule is not equivalent to a run whose schedule ends at 5,000.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment0/03_budget_pilots')
report = cp.NotebookReport('Experiment 0: decide how long adaptation should run')


## 1. Budget-pilot coverage

One reference recipe per base/task on the first partition. These are operational and descriptive pilots, not independent confirmation datasets.

In [ ]:
budgets = [cp.load_campaign(0, track, 'budget') for track in ('pd','lgd')]
for budget in budgets:
    cp.show(sink, cp.plot_coverage(budget))
report.add('1. Budget-pilot coverage', '\n\n'.join(cp.coverage_summary(c) for c in budgets))

## 2. Behavior across update milestones

Curves are paired with the same table at update zero. Positive means higher AUC for PD or fractional RMSE reduction for LGD.

In [ ]:
for budget in budgets:
    cp.show(sink, cp.plot_trajectory_pages(budget))
report.add('2. Update-indexed behavior', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.effects(c)) for c in budgets))

## 3. Behavior against row exposure

The same update budget need not correspond to the same number of processed rows across bases and tables. Exposures include repeated rows and do not represent unique examples.

In [ ]:
for budget in budgets:
    cp.show(sink, cp.plot_trajectory_pages(budget, x='processed_rows'))
report.add('3. Row exposure', '\n\n'.join(c.track.upper()+'\n'+(cp.effects(c).groupby(['base','updates']).processed_rows.median().to_string() if not cp.effects(c).empty else 'No measurements.') for c in budgets))

## 4. Transfer and cost

Compare changes on training and held-out tables without confusing their different initial difficulty with an effect of training. Inspect compute cost alongside behavior.

In [ ]:
for budget in budgets:
    cp.show(sink, cp.plot_train_test(budget))
    cp.show(sink, cp.plot_optimization(budget, 'train_loss'))
    cp.show(sink, cp.plot_diagnostics(budget))
report.add('4. Transfer and cost', 'Seen and held-out effects are separately paired to update zero. Their table sets differ; this is a transfer diagnostic, not an IID generalization-gap estimate.')

## 5. Freeze the campaign horizon

Record the budget decision before preparing the large-run plans. Keep main, seed and sampling budgets aligned, while retaining their explicitly different sampling policies.

In [ ]:
report.add('5. Horizon decision', 'Provisional main target: 5,000 successful updates. The pilot does not automatically overwrite configs, select a checkpoint or submit jobs.')

## 6. Non-credit retention

Fixed public non-credit datasets remain outside the adaptation corpus. Curves compare every milestone with the same starting checkpoint, rows and monitoring seed. This small panel measures retention on those tables; it does not establish universal absence of forgetting.

In [ ]:
from src.visualize import diagnostics as dg
for current in budgets:
    cp.show(sink, cp.plot_trajectory_pages(current, split='ood'))
report.add('6. Non-credit retention', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.endpoint_effects(c, 'ood')) for c in budgets))

## 7. Parameter movement and sampled resources

Milestone tensor summaries complement total norm-weighted drift. Resource plots use one median per trial and omit unavailable counters; sampled device utilization is not precise kernel time or energy.

In [ ]:
for current in budgets:
    cp.show(sink, dg.plot_parameters(current))
    cp.show(sink, dg.plot_resources(current))
report.add('7. Parameters and resources', '\n\n'.join(dg.summary(c) for c in budgets))

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))